# Analyse temporelle des clusters


In [ ]:
import os
import glob
import s3fs

import pandas as pd
import csv
import matplotlib.pyplot as plt
import numpy as np


import datetime
import json
import re
import tqdm


# Chargement des fichiers

In [ ]:
# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"
FILE_KEY_S3 = "data_pest/"
FILE_PATH_S3 = BUCKET_OUT + "/" + FILE_KEY_S3

In [ ]:
i = -1
for file in fs.ls(BUCKET_OUT + "/" + FILE_KEY_S3):
    if ".csv" in file :
        dict_id = {}
        file_path = BUCKET_OUT + "/" + FILE_KEY_S3 + "/" + file
        print(file)
        i+=1
        with fs.open(file, mode="r", newline = '') as csv_in:
            reader = csv.DictReader(csv_in)
            headers = reader.fieldnames
            for x in headers:
                if "id" in x:
                    dict_id[x] = str
        with fs.open(file, mode = "rb") as file_in:
            df0 = pd.read_csv(file_in, sep =",", dtype = dict_id, usecols=['id','user_id', 'text', 'user_screen_name', 'to_userid',
       'to_tweetid','mentioned_names', 'mentioned_ids','local_time'])

        if i > 0 :
            df = pd.concat([df, df0])
        else:
            df = df0.copy()
            


In [ ]:

tweets= df.drop_duplicates()
tweets['local_time'] = pd.to_datetime(tweets['local_time'])
#tweets


print (len(tweets))

In [ ]:
def upload_csvfile(file_path, cols_to_keep=None):
    dict_id = {}
    with fs.open(file_path, mode="r", newline = '') as csv_in:
        reader = csv.DictReader(csv_in)
        headers = reader.fieldnames
        for x in headers:
            if "id" in x:
                dict_id[x] = str
    with fs.open(file_path, mode = "rb") as file_in:
        data = pd.read_csv(file_in, sep =",", dtype = dict_id, usecols= cols_to_keep)

    return data


    

In [ ]:
FILE_KEY_S3 = "data_pest/industry"
name_file = "20250902_pesticides_industries.csv"



file_path = BUCKET_OUT + "/" + FILE_KEY_S3 + "/" + name_file
print(file_path)

users0 = upload_csvfile(file_path, cols_to_keep=None)
users0


dict_col = {}
            
for x in ['field_work_x', 'metier_x', 'Ingenieur_x', 'Agronome_x',
       'industry_verif_x', 'field_work_y', 'metier_y', 'Ingenieur_y',
       'Agronome_y', 'industry_verif_y']:
    dict_col[x] = x.replace("_x","").replace('_y',"")

users0 = users0[['user_id', 'user_screen_name', 'user_description',
       'account_publication', 'industry', 'Type_entite', 'company', 'UIPP',
       'field_work_x', 'metier_x', 'Ingenieur_x', 'Agronome_x',
       'industry_verif_x']].rename(columns=dict_col)

In [ ]:
FILE_KEY_S3 = "data_pest/users"
name_file = "the_true_all_annotated_account.csv"



file_path = BUCKET_OUT + "/" + FILE_KEY_S3 + "/" + name_file
print(file_path)

users1 = upload_csvfile(file_path, cols_to_keep=None)

users1["user_description"] = users1.user_description.str.replace("\n", " ")
users1["user_description"] = users1.user_description.str.replace(",", " ")
#users1.to_csv(f"{path_base}the_true_all_annotated_account.csv", sep = ",", index = False)
#users1 = users1.drop_duplicates().loc[users1["platform"]=="twitter"]
users1 = users1[['user_id', 'user_screen_name', 'user_name', 'Type_entite',
       'User_role', 'user_description', 'account_publication', 
       'Synthetic_logics']].drop_duplicates()#.reset_index()


In [ ]:
keep_all = True #Pour garder ou pas les comptes d'industries dont on a pas collecté les retweets

list_col = ['user_id', 'user_screen_name',
       'user_description', 'account_publication', 'Type_entite']

if keep_all == True:
    users1b = users1.merge(users0, on = list_col, how = "left")
    already_in_users1 = [x for x in users1b.user_id.unique()]
    users2 = users0.loc[~users0["user_id"].isin(already_in_users1)]
    #print(len(users), len(users0))
    
    users = pd.concat([users1b, users2], ignore_index=True)
else:
    users = users1.merge(users0, on = list_col, how = "left")
    


len(users)

In [ ]:
users1

In [ ]:
users.loc[(users["industry_verif"].isna()), "industry"]=False
users.loc[(users["industry_verif"] == True), "industry"]=True
users.loc[(users["industry"].isna()) | (users["industry"]==False) , "company"]="irrelevent"
users.loc[(users["industry"]==True) & (users["company"].isna()) , "company"]= "Unknown"



users = users.drop_duplicates(["user_screen_name","user_id","industry", "Type_entite"])

dict_logic = dict(zip(users.user_id, users.Synthetic_logics))
users["Synthetic_logics2"] = users.user_id.map(dict_logic.get)
users.loc[(users.industry==True), "Synthetic_logics2"] = "industry_pesticide"

dict_translation = {'industry_pesticide':'Industries pesticides', 
              'Agricultural Practices':'Pratiques agricoles',
       'Agroindustrial_perspectives':'Perspectives agroindustrielles', 
              'Comment_the_news':'Commentaires de l\'actualité',
       'Ecological_perspectives':'Perspectives écologiques', 
              'Health_perspectives':'Perspectives sanitaires',
       'Marketing_logic':'Perspectives commerciales', 
              'Pesticide_free_agriculture':'Lutte contre les pesticides',
       'Rationalist_perspectives':'Perspectives rationalistes'}

users["Synthetic_logicsfr"]= users.Synthetic_logics2.map(dict_translation.get)


In [ ]:
dict_logic2 = dict(zip(users.user_id, users.Synthetic_logics2))

#tweets["to_username"] = tweets.to_userid.map(dict(zip(users.user_id, users.user_screen_name)).get)

tweets["logic"] = tweets.user_id.map(dict_logic2.get)

tweets["company"] = tweets.user_id.map(dict(zip(users.user_id, users.company)).get)

tweets["industry"] = tweets.user_id.map(dict(zip(users.user_id, users.industry_verif)).get)

In [ ]:
from search_regex import queries_in_dictionnary, texts_in_dictionnaries, search_regex

In [ ]:
dict_text = dict(zip(tweets.id, tweets.text.str.lower()))

FILE_KEY_S3 = "data_pest/queries"
name_file = "list_of_queries.csv"

file_path = BUCKET_OUT + "/" + FILE_KEY_S3 + "/" + name_file


queries = upload_csvfile(file_path)


dict_queries = queries_in_dictionnary(data = queries, 
                                      column_name_of_queries = "nom_cluster.3", 
                                      list_of_regex = ["query_1"])

dict_queries

In [ ]:
d, dict_match_queries = search_regex(dict_queries = dict_queries, dict_text = dict_text)
d["somme"] = d.sum(axis=1)

In [ ]:
dict_match_queries

In [ ]:
d["id"] = d.index
list_column = list(d.columns)[0:-2]
list_column

In [ ]:
users.columns

In [ ]:
d["logic"] = d.id.map(dict(zip(tweets.id, tweets.logic)).get)
d["user_id"] = d.id.map(dict(zip(tweets.id, tweets.user_id)).get)
d["user_screen_name"] = d.id.map(dict(zip(tweets.id, tweets.user_screen_name)).get)
d["industry"] = d.id.map(dict(zip(tweets.id, tweets.industry)).get)
d["Type_entite"] = d.user_id.map(dict(zip(users.user_id, users.Type_entite)).get)
d["firm"] = d.user_id.map(dict(zip(users.user_id, users.company)).get)

np.sum(d['La mortalité des abeilles'].loc[~d.logic.isna()])

In [ ]:
dict_date = dict(zip(tweets.id, tweets.local_time))
d["local_time"] = d.id.map(dict_date.get)


## Tri des cluster en fonction des années

Les industries pesticides émergent tard, possible que le faible engagement sur certains thèmes s'explique par leur absence.

In [ ]:
d1 = d.loc[(d.somme>=0) & ~(d.logic.isna())]
d1_industry = d1.loc[d1.industry==True]
d1_industry.local_time.loc[d1.local_time > "2016-01-01"].min()

In [ ]:
def tracer_graphique(data, var_time= "local_time", var_group ="logic", value= "id", d="Mois",  fonction="size", title="Corpus", save_fig=None):
    scale = {"Année":"YE","Mois":"ME","Jour":"d"}
    df1 = data.groupby([var_group, var_time]).agg(nb=(value, fonction)).reset_index()
    fig,ax=plt.subplots(1,figsize=(10, 8))

    #df["num"] = 1 # valeur par article pour comptage
    leg = []
    
    if var_group in data.columns:
        for i, j in data.groupby(var_group):
            leg.append(i)
            if fonction == "size":
                j.set_index(var_time)[value].resample(scale[d]).size().plot(ax=ax,style="-")
            elif fonction == "sum":
                j.set_index(var_time)[value].resample(scale[d]).sum().plot(ax=ax,style="-")
        plt.legend(leg)
    else:
        #data.set_index("date")["id"].resample(scale[d]).size().plot(ax=ax,style=".-")
        plt.plot(df1.local_time, df1.nb)
    plt.title(f"Distribution par {d.lower()} du corpus {title}")
    #plt.xlabel("Temps (par %s)"%d)
    plt.ylabel("Nombre de tweets")
    plt.tight_layout()
    
    if save_fig is not None:
        plt.savefig(f"{save_fig}.pdf", dpi= 300)
        fig.savefig(f"{save_fig}.jpeg", dpi=100, format = "jpeg")
    else:
        pass
    
    return fig

In [ ]:

d2 = d1_industry.loc[(d1_industry["local_time"]>='2010-06-08') & (d1_industry["local_time"]<='2021-09-10') & (d1_industry.Type_entite=="Personne")]
corpus = "industries pesticides"
fig = tracer_graphique(d2, var_group = "firm", value="SDHI", fonction="sum", d = "Mois", title = corpus, save_fig=None)

d2.groupby(["user_screen_name","firm"]).agg(nb=('SDHI',"sum")).reset_index().sort_values("nb")

In [ ]:
d_sdhi = d2.loc[(d2.SDHI==1)&(d2.local_time>="2018-01-01")].sort_values("local_time")

In [ ]:
for n, t in enumerate(d_sdhi.id):
    print("-----------------\n")
    print(f"Date : {d_sdhi.local_time.iloc[n]}\n")
    print(f"logic: {d_sdhi.logic.iloc[n]}\n")
    print(f"Nom: {d_sdhi.user_screen_name.iloc[n]}\n")
    print(tweets.text.loc[tweets.id==t].values)

In [ ]:
print(tweets.text.loc[tweets.id=="981954985859665921"].values)

In [ ]:
f_sdhi = d.loc[(d.local_time >='2018-04-01') &(d.local_time<'2019-01-01')].groupby(["logic"]).agg(nb = ('SDHI', "sum")).reset_index()
f_sdhi["freq"] = f_sdhi.nb / np.sum(f_sdhi.nb)*100
f_sdhi["total"] = np.sum(f_sdhi.nb)
f_sdhi.sort_values("freq", ascending=False)


In [ ]:

f_sdhi_user = d.loc[d.SDHI==1].drop_duplicates("user_id").groupby(["logic"]).agg(nb=('SDHI',"sum")).reset_index().sort_values("nb")
f_sdhi_user["freq"] = f_sdhi_user.nb / np.sum(f_sdhi_user.nb)*100
f_sdhi_user.sort_values("freq", ascending=False)

In [ ]:
f_sdhi1 = f_sdhi.merge(f_sdhi_user, on = ["logic"], how = "left")
f_sdhi1["ratio"] = f_sdhi1.nb_x/f_sdhi1.nb_y
f_sdhi1.sort_values("ratio")

In [ ]:
d.loc[(d.logic=="industry_pesticide")].groupby(["user_screen_name"]).agg(nb=('Néonicotinoïdes',"sum")).reset_index().sort_values("nb").tail(30)

In [ ]:
d2 = d.loc[d.logic.isin(['Health_perspectives', 'Pesticide_free_agriculture',
       'Ecological_perspectives', 'Agroindustrial_perspectives','Comment_the_news',
       'Rationalist_perspectives',
       'industry_pesticide']) & (d.somme>0)]
d2.logic.unique()

In [ ]:
d.columns

In [ ]:
list_columns = ['Le vin', 'Plan écophyto', 'Transition écologique et débats publics',
       'Épandages et riverains', 'Le rapport du CIRC', 'SDHI',
       'Affaire Triskalia', 'Les espèces nuisibles', 'Le virus Zika',
       'Chlordécone', 'La mortalité des abeilles', 'Néonicotinoïdes',
       'Procès de Monsanto', 'La science en question',
       'Politiques agricoles françaises', 'La crise de la biodiversité',
       'Perturbateurs endocriniens', 'Pollution de l\'eau',
       'Sécurité des aliments', 'Réchauffement climatique',
       'Régulation européenne', 'Troubles de santé', 'Glyphosate', 
       'Glyphosate2']

In [ ]:
i=-1
for n, x in enumerate(d2.logic.unique()):
    i+=1
    list_size = []
    list_name_cluster= []
    for c in list_columns:
        list_size.append(np.sum(d2[c].loc[d2.logic == x]))
        list_name_cluster.append(c)
    data = {"cluster": list_name_cluster, "nb":list_size}
    dtopic0 = pd.DataFrame(data)
    dtopic0["logic"] = x
    if i == 0:
        dtopic = dtopic0.copy()
    else:
        dtopic = pd.concat([dtopic,dtopic0])
    


i = -1
df1 = tweets.loc[tweets.logic.isin(['Health_perspectives', 'Pesticide_free_agriculture',
       'Ecological_perspectives', 'Agroindustrial_perspectives','Comment_the_news',
       'Rationalist_perspectives',
       'industry_pesticide'])].groupby(["logic"])["logic"].count().reset_index(name='count')
df1

In [ ]:
dtopic.loc[dtopic.logic=="industry_pesticide"].sort_values("nb")